# 06 - Returns Regression and Hybrid Inference
Modelado de retornos porcentuales, calibracion de clasificacion y sistema de inferencia hibrido DATAGIA V1.

In [1]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

from sklearn.calibration import CalibratedClassifierCV
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier

base_dir = Path.cwd()
for root in [base_dir, *base_dir.parents]:
    if (root / "src").exists():
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))
        break

from src.data_processing.build_dataset import get_training_features

warnings.filterwarnings("ignore")

dataset_path = None
for root in [base_dir, *base_dir.parents]:
    candidate = root / "data" / "processed" / "dataset_entrenamiento_final.csv"
    if candidate.exists():
        dataset_path = candidate
        break
if dataset_path is None:
    raise FileNotFoundError("dataset_entrenamiento_final.csv not found under data/processed")

df = pd.read_csv(dataset_path, parse_dates=["date"])
print("dataset:", dataset_path.resolve())
print("shape:", df.shape)

blacklist = [
    "precio_provincial_lag_1"
    "precio_provincial_lag_2"
    "precio_provincial_lag_3"
    "precio_vecinos_media_lag1"
    "precio_nacional_base_ma3"
    "precio_nacional_base_ma6"
    "precio_nacional_base_vol3"
    "precio_nacional_base_vol6"
]
target_candidates = ["precio_provincial_TARGET_H1", "precio_provincial_TARGET_H2", "precio_provincial_TARGET_H3"]
missing_targets = [t for t in target_candidates if t not in df.columns]
if missing_targets:
    raise ValueError(f"Missing target columns: {missing_targets}")
available_targets = target_candidates
horizons = [1, 2, 3]

split_date = pd.Timestamp("2021-01-01")
train_mask = df["date"] < split_date
test_mask = ~train_mask

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()
print("train rows:", train_df.shape[0], "test rows:", test_df.shape[0])

identifiers = ["date", "provincia", "cereal_predominante"]
training_cols = get_training_features(df)
feature_cols = [
    c for c in training_cols
    if c in df.columns and c not in identifiers + available_targets
]
feature_cols = [c for c in feature_cols if c not in blacklist]

X_full = df[feature_cols].copy()
bool_cols = X_full.select_dtypes(include=["bool"]).columns
if len(bool_cols) > 0:
    X_full[bool_cols] = X_full[bool_cols].astype(int)

cat_cols = X_full.select_dtypes(include=["object", "category"]).columns.tolist()
if cat_cols:
    X_full = pd.get_dummies(X_full, columns=cat_cols, drop_first=False)

X_train = X_full.loc[train_mask].copy()
X_test = X_full.loc[test_mask].copy()
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

base_price_col = "precio_provincial_lag_1"
if base_price_col not in df.columns:
    raise ValueError("precio_provincial_lag_1 missing for targets")

def build_targets(horizon: int):
    target_reg = f"precio_provincial_TARGET_H{horizon}"
    y_train_reg = train_df[target_reg]
    y_test_reg = test_df[target_reg]
    base_train = train_df[base_price_col]
    base_test = test_df[base_price_col]
    y_train_clf = (y_train_reg - base_train > 0).astype(int)
    y_test_clf = (y_test_reg - base_test > 0).astype(int)
    return y_train_reg, y_test_reg, y_train_clf, y_test_clf, base_train, base_test

def regression_metrics(y_true, y_pred):
    aligned = pd.concat([y_true, y_pred], axis=1).dropna()
    if aligned.empty:
        return {"MAE": np.nan, "RMSE": np.nan, "Pearson": np.nan}
    y_true_clean = aligned.iloc[:, 0]
    y_pred_clean = aligned.iloc[:, 1]
    mae = mean_absolute_error(y_true_clean, y_pred_clean)
    rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    pearson = pearsonr(y_true_clean, y_pred_clean)[0] if y_true_clean.nunique() > 1 else np.nan
    return {"MAE": float(mae), "RMSE": float(rmse), "Pearson": float(pearson) if pearson == pearson else np.nan}

def classification_metrics(y_true, proba, pred):
    acc = accuracy_score(y_true, pred)
    auc = roc_auc_score(y_true, proba) if y_true.nunique() > 1 else np.nan
    return {"DA": float(acc), "AUC": float(auc) if auc == auc else np.nan}

def top_features_from_model(model, feature_names, top_k=5):
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        order = np.argsort(importances)[::-1][:top_k]
        return [feature_names[i] for i in order]
    if hasattr(model, "coef_"):
        coefs = np.ravel(model.coef_)
        order = np.argsort(np.abs(coefs))[::-1][:top_k]
        return [feature_names[i] for i in order]
    return []

dataset: C:\Users\marco\Desktop\Repos\DATAGIA-21\data\processed\dataset_entrenamiento_final.csv
shape: (7047, 78)
train rows: 5394 test rows: 1653


## 1. Torneo de regresion por retornos (H1-H3)
Targets: RETURN_Hx = (TARGET_Hx - precio_provincial_lag_1) / precio_provincial_lag_1.

In [3]:
tscv = TimeSeriesSplit(n_splits=5)

return_champions = {}
return_results = []
return_top5 = {}

reg_param_grids = {
    "Ridge": {
        "model__alpha": [0.1, 1.0, 5.0, 10.0, 25.0],
    },
    "RF": {
        "model__n_estimators": [300, 500, 700],
        "model__max_depth": [4, 6, 8, 12, None],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", 0.6, 0.8],
    },
    "XGB": {
        "model__n_estimators": [200, 400, 600],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.6, 0.8, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
        "model__min_child_weight": [1, 5, 10],
    },
}

for h in horizons:
    y_train_reg, y_test_reg, _, _, base_train, base_test = build_targets(h)
    train_mask_h = base_train.notna() & (base_train != 0) & y_train_reg.notna()
    test_mask_h = base_test.notna() & (base_test != 0) & y_test_reg.notna()
    X_train_h = X_train.loc[train_mask_h]
    X_test_h = X_test.loc[test_mask_h]
    base_train_h = base_train.loc[train_mask_h]
    base_test_h = base_test.loc[test_mask_h]

    y_train_ret = (y_train_reg.loc[train_mask_h] - base_train_h) / base_train_h
    y_test_ret = (y_test_reg.loc[test_mask_h] - base_test_h) / base_test_h

    reg_models = {
        "Ridge": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(random_state=42)),
        ]),
        "RF": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),
        ]),
        "XGB": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBRegressor(random_state=42, n_jobs=-1, objective="reg:squarederror")),
        ]),
    }

    best_by_name = {}
    for name, model in reg_models.items():
        search = RandomizedSearchCV(
            model,
            param_distributions=reg_param_grids[name],
            n_iter=20,
            scoring="neg_mean_absolute_error",
            cv=tscv,
            random_state=42,
            n_jobs=-1,
        )
        search.fit(X_train_h, y_train_ret)
        best_by_name[name] = search.best_estimator_

    best_name = None
    best_metrics = None
    best_model = None
    for name, model in best_by_name.items():
        preds = pd.Series(model.predict(X_test_h), index=y_test_ret.index)
        metrics = regression_metrics(y_test_ret, preds)
        return_results.append({"horizon": h, "model": name, **metrics})
        if best_metrics is None or metrics["MAE"] < best_metrics["MAE"]:
            best_name = name
            best_metrics = metrics
            best_model = model

    baseline_pred = pd.Series(0.0, index=y_test_ret.index)
    baseline_metrics = regression_metrics(y_test_ret, baseline_pred)
    return_champions[h] = {
        "model": best_name,
        "metrics": best_metrics,
        "baseline": baseline_metrics,
        "estimator": best_model,
        "y_test_ret": y_test_ret,
        "X_test": X_test_h,
    }
    return_top5[h] = top_features_from_model(best_model.named_steps["model"], X_train.columns.tolist(), top_k=5)

return_results_df = pd.DataFrame(return_results)
return_results_df

,horizon,model,MAE,RMSE,Pearson
0,1,Ridge,0.102331,0.132050,0.399207
1,1,RF,0.046730,0.066839,0.563457
2,1,XGB,0.048731,0.067212,0.488046
3,2,Ridge,0.140471,0.179085,0.331040
4,2,RF,0.059780,0.089101,0.672430
5,2,XGB,0.070098,0.098849,0.400684
6,3,Ridge,0.147624,0.178648,0.480131
7,3,RF,0.073652,0.110330,0.690332
8,3,XGB,0.096136,0.133146,0.233727


## 2. Clasificador calibrado (H1-H3)
Se calibra con sigmoid y se conserva el mejor modelo por DA.

In [5]:
clf_param_grids = {
    "RF": {
        "model__n_estimators": [300, 500, 700],
        "model__max_depth": [4, 6, 8, 12, None],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", 0.6, 0.8],
    },
    "XGB": {
        "model__n_estimators": [200, 400, 600],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.6, 0.8, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
        "model__min_child_weight": [1, 5, 10],
    },
}

clf_champions = {}
clf_calibrated = {}

for h in horizons:
    _, _, y_train_clf, y_test_clf, _, _ = build_targets(h)
    train_mask_c = y_train_clf.notna()
    test_mask_c = y_test_clf.notna()
    X_train_c = X_train.loc[train_mask_c]
    X_test_c = X_test.loc[test_mask_c]
    y_train_c = y_train_clf.loc[train_mask_c]
    y_test_c = y_test_clf.loc[test_mask_c]

    clf_models = {
        "RF": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(random_state=42, n_jobs=-1)),
        ]),
        "XGB": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(random_state=42, n_jobs=-1, eval_metric="logloss")),
        ]),
    }

    best_by_name = {}
    for name, model in clf_models.items():
        search = RandomizedSearchCV(
            model,
            param_distributions=clf_param_grids[name],
            n_iter=20,
            scoring="roc_auc",
            cv=tscv,
            random_state=42,
            n_jobs=-1,
        )
        search.fit(X_train_c, y_train_c)
        best_by_name[name] = search.best_estimator_

    best_name = None
    best_metrics = None
    best_model = None
    for name, model in best_by_name.items():
        proba = model.predict_proba(X_test_c)[:, 1]
        pred = (proba >= 0.5).astype(int)
        metrics = classification_metrics(y_test_c, proba, pred)
        if best_metrics is None or metrics["DA"] > best_metrics["DA"]:
            best_name = name
            best_metrics = metrics
            best_model = model

    clf_champions[h] = {
        "model": best_name,
        "metrics": best_metrics,
        "estimator": best_model,
    }

    calib = CalibratedClassifierCV(estimator=best_model, method="sigmoid", cv=TimeSeriesSplit(n_splits=3))
    calib.fit(X_train_c, y_train_c)
    proba_cal = calib.predict_proba(X_test_c)[:, 1]
    pred_cal = (proba_cal >= 0.5).astype(int)
    metrics_cal = classification_metrics(y_test_c, proba_cal, pred_cal)
    clf_calibrated[h] = {
        "model": best_name,
        "metrics": metrics_cal,
        "estimator": calib,
        "y_test": y_test_c,
        "X_test": X_test_c,
    }

print("Clf calibrated AUC:", {h: clf_calibrated[h]["metrics"]["AUC"] for h in horizons})

Clf calibrated AUC: {1: 0.8576100562401934, 2: 0.8200274725274725, 3: 0.8702313832840836}


## 3. Inferencia hibrida DATAGIA V1
Regla: prob>0.65 y |retorno|>1%. Se evalua DA en el subset con senal.

In [6]:
hybrid_rows = []

for h in horizons:
    y_test_c = clf_calibrated[h]["y_test"]
    X_test_c = clf_calibrated[h]["X_test"]
    proba = clf_calibrated[h]["estimator"].predict_proba(X_test_c)[:, 1]

    ret_model = return_champions[h]["estimator"]
    y_test_ret = return_champions[h]["y_test_ret"]
    X_test_ret = return_champions[h]["X_test"]
    ret_pred = pd.Series(ret_model.predict(X_test_ret), index=y_test_ret.index)

    idx = y_test_c.index.intersection(ret_pred.index)
    proba_s = pd.Series(proba, index=y_test_c.index).loc[idx]
    y_true_dir = y_test_c.loc[idx]
    ret_pred_s = ret_pred.loc[idx]

    buy_mask = (proba_s >= 0.65) & (ret_pred_s >= 0.01)
    sell_mask = (proba_s <= 0.35) & (ret_pred_s <= -0.01)
    signal_mask = buy_mask | sell_mask

    pred_dir = pd.Series(np.nan, index=idx)
    pred_dir[buy_mask] = 1
    pred_dir[sell_mask] = 0

    if signal_mask.sum() > 0:
        da_hybrid = accuracy_score(y_true_dir.loc[signal_mask], pred_dir.loc[signal_mask])
    else:
        da_hybrid = np.nan

    # Baselines on same signal subset
    if signal_mask.sum() > 0:
        y_subset = y_true_dir.loc[signal_mask]
        da_persist = accuracy_score(y_subset, np.zeros_like(y_subset))
        da_up = accuracy_score(y_subset, np.ones_like(y_subset))
        da_down = accuracy_score(y_subset, np.zeros_like(y_subset))
    else:
        da_persist = np.nan
        da_up = np.nan
        da_down = np.nan

    hybrid_rows.append({
        "horizon": h,
        "signals": int(signal_mask.sum()),
        "DA_hybrid": da_hybrid,
        "DA_persist": da_persist,
        "DA_solo_sube": da_up,
        "DA_solo_baja": da_down,
    })

hybrid_df = pd.DataFrame(hybrid_rows)
hybrid_df

,horizon,signals,DA_hybrid,DA_persist,DA_solo_sube,DA_solo_baja
0,1,0,NaN,NaN,NaN,NaN
1,2,4,1.0,0.00000,1.00000,0.00000
2,3,92,1.0,0.01087,0.98913,0.01087


## 4. Reporte maestro
Genera SISTEMA_INFERENCIA_DATAGIA_V1.md con benchmarks y top 5.

In [8]:
report_root = None
for root in [base_dir, *base_dir.parents]:
    candidate = root / "reports"
    if candidate.exists():
        report_root = candidate
        break
if report_root is None:
    report_root = base_dir / "reports"
    report_root.mkdir(parents=True, exist_ok=True)

report_path = report_root / "SISTEMA_INFERENCIA_DATAGIA_V1.md"

return_benchmark_rows = []
for h in horizons:
    champ = return_champions[h]
    beats = champ["metrics"]["MAE"] < champ["baseline"]["MAE"]
    return_benchmark_rows.append({
        "horizon": h,
        "model": champ["model"],
        "MAE_model": champ["metrics"]["MAE"],
        "RMSE_model": champ["metrics"]["RMSE"],
        "MAE_baseline": champ["baseline"]["MAE"],
        "RMSE_baseline": champ["baseline"]["RMSE"],
        "beats_baseline": beats,
    })

top5_rows = []
for h in horizons:
    top5_rows.append({
        "horizon": h,
        "model": return_champions[h]["model"],
        "Top5": ", ".join(return_top5[h]),
    })

lines = [
    "# SISTEMA_INFERENCIA_DATAGIA_V1",
    "",
    "## Resumen",
    "Evaluacion de regresion por retornos y sistema hibrido (clasificacion calibrada + regresor de retornos).",
    "Regla de inferencia: prob>0.65 y |retorno|>1%.",
    "",
    "## Benchmark de retornos (vs retorno cero)",
    pd.DataFrame(return_benchmark_rows).to_markdown(index=False),
    "",
    "## Desempeno hibrido (DA en subset con senal)",
    hybrid_df.to_markdown(index=False),
    "",
    "## Top 5 variables que explican retornos",
    pd.DataFrame(top5_rows).to_markdown(index=False),
    "",
]

report_path.write_text("\n".join(lines), encoding="utf-8")
print("Reporte guardado en:", report_path.resolve())

Reporte guardado en: C:\Users\marco\Desktop\Repos\DATAGIA-21\notebooks\training\reports\SISTEMA_INFERENCIA_DATAGIA_V1.md
